<a href="https://colab.research.google.com/github/DJCordhose/buch-machine-learning-notebooks/blob/master/kap10-bert-tasks-pytorch-v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT Transformer Tasks

* Mehr Beispiele: https://huggingface.co/transformers/task_summary.html

## Der Transformer Zoo
![Encoder-Decoder](https://raw.githubusercontent.com/DJCordhose/ml-resources/main/img/transformers/transformer-encoder-decoder.png)

## BERT - das typische Encoder-Modell

Kleinere Modelle, können typischerweise auch auf CPU ausgeführt werden

BERT verstehen: https://pair.withgoogle.com/explorables/fill-in-the-blank/

In [1]:
# !nvidia-smi

In [2]:
import sys
IN_COLAB = 'google.colab' in sys.modules
IN_COLAB

False

In [3]:
# if IN_COLAB:
#     # https://huggingface.co/docs/transformers/installation
#     !pip install -q transformers torch


In [4]:
import torch
torch.__version__


'2.11.0'

In [5]:
import transformers
transformers.__version__

'5.6.2'

# Transformer Pipelines

### Pipelines are made of:

- A [tokenizer](tokenizer) in charge of mapping raw textual input to token.
- A [model](model) to make predictions from the inputs.
- Some (optional) post processing for enhancing model's output.

### Available Pipeline Tasks

Die verfügbaren Pipeline-Tasks hängen von der installierten `transformers`-Version ab. In `transformers` 5.x ist `question-answering` nicht mehr als Pipeline-Task registriert; der Abschnitt unten verwendet deshalb das PyTorch-Modell direkt.

In [6]:
# Alle verfügbaren Tasks ausgeben:
from transformers.pipelines import SUPPORTED_TASKS

for task, config in sorted(SUPPORTED_TASKS.items()):
    pipeline_class = config.get("impl")
    pipeline_name = pipeline_class.__name__ if pipeline_class else "unbekannte Pipeline"

    print(f'- "{task}": will return a [{pipeline_name}].')

- "any-to-any": will return a [AnyToAnyPipeline].
- "audio-classification": will return a [AudioClassificationPipeline].
- "automatic-speech-recognition": will return a [AutomaticSpeechRecognitionPipeline].
- "depth-estimation": will return a [DepthEstimationPipeline].
- "document-question-answering": will return a [DocumentQuestionAnsweringPipeline].
- "feature-extraction": will return a [FeatureExtractionPipeline].
- "fill-mask": will return a [FillMaskPipeline].
- "image-classification": will return a [ImageClassificationPipeline].
- "image-feature-extraction": will return a [ImageFeatureExtractionPipeline].
- "image-segmentation": will return a [ImageSegmentationPipeline].
- "image-text-to-text": will return a [ImageTextToTextPipeline].
- "keypoint-matching": will return a [KeypointMatchingPipeline].
- "mask-generation": will return a [MaskGenerationPipeline].
- "object-detection": will return a [ObjectDetectionPipeline].
- "table-question-answering": will return a [TableQuestionAn

## Sentiment Analysis

model trained on the glue dataset: https://huggingface.co/datasets/glue


In [ ]:
from transformers import pipeline

# Kein Modellname angegeben: Pipeline wählt automatisch.
classifier = pipeline(task="sentiment-analysis")
classifier.model.name_or_path

# ...oder alles explizit angeben:
# sentiment_model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# classifier = pipeline(
#     task="sentiment-analysis",
#     model=sentiment_model_name,
#     tokenizer=sentiment_model_name,
#     framework="pt"
# )
# classifier.model.name_or_path

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

'distilbert/distilbert-base-uncased-finetuned-sst-2-english'

In [ ]:
# Ein erster Versuch:
classifier("I hate you")

[{'label': 'NEGATIVE', 'score': 0.9991129040718079}]

In [ ]:
# ...und noch einer...
classifier("I love you")

[{'label': 'POSITIVE', 'score': 0.9998656511306763}]

In [ ]:
# https://huggingface.co/models?pipeline_tag=text-classification&language=de&sort=trending

# Deutsches Modell verwenden:
classifier = pipeline(
    task="sentiment-analysis",
    model="oliverguhr/german-sentiment-bert",
    tokenizer="oliverguhr/german-sentiment-bert",
    framework="pt"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Weitere Tests

In [39]:
classifier("Ich liebe dich")

[{'label': 'positive', 'score': 0.9846151471138}]

In [40]:
classifier("Ick liebe dir")

[{'label': 'positive', 'score': 0.9546654224395752}]

In [41]:
classifier("Ich bin mit diesem Artikel sehr zufrieden")

[{'label': 'positive', 'score': 0.9964079260826111}]

In [42]:
classifier("Ich bin mit diesem Artikel nicht unzufrieden")

[{'label': 'positive', 'score': 0.9832236766815186}]

In [43]:
classifier("Ich bin mit diesem Artikel zufrieden (von wegen)")

[{'label': 'positive', 'score': 0.978872537612915}]

In [16]:
classifier("Ich mag dich nicht")

[{'label': 'negative', 'score': 0.9946463704109192}]

## Extractive Question Answering

* model: https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad
* dataset: The Stanford Question Answering Dataset (SQuAD) is a collection of question-answer pairs derived from Wikipedia articles. In SQuAD, the correct answers of questions can be any sequence of tokens in the given text.
  * https://paperswithcode.com/dataset/squad
  * https://huggingface.co/datasets/squad
  * https://rajpurkar.github.io/SQuAD-explorer/

In `transformers` 5.x ist `question-answering` nicht mehr als Pipeline-Task registriert. Wir verwenden deshalb das PyTorch-Modell direkt und bilden die Pipeline-Funktionalität in einer kleinen Hilfsfunktion nach.

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

qa_model_name = "distilbert/distilbert-base-cased-distilled-squad"

qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_model.eval()

qa_model.name_or_path

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

'distilbert/distilbert-base-cased-distilled-squad'

In [18]:
context = r"""
Extractive Question Answering is the task of extracting an answer from a text given a question. An example of a
question answering dataset is the SQuAD dataset, which is entirely based on that task. If you would like to fine-tune
a model on a SQuAD task, you may leverage the examples/pytorch/question-answering/run_squad.py script.
"""

In [19]:
# In Transformers 4 gab es für diesen Task noch die bequeme Pipeline
# pipeline("question-answering"). In Transformers 5 ist diese Pipeline
# nicht mehr verfügbar. Die Aufgabe selbst ist aber weiterhin sinnvoll:
#
# Ein Question-Answering-Modell liefert Start- und Endpositionen der
# Antwort im Kontext. Diese kleine Hilfsfunktion bildet daher das frühere
# Pipeline-Verhalten nach, damit der restliche Notebook-Code und die
# Ausgaben möglichst nah an der Vorgängerauflage bleiben.

import torch

def question_answerer(question, context, max_answer_length=30):
    # Der Tokenizer kodiert Frage und Kontext gemeinsam.
    # sequence_ids merkt sich dabei, welche Tokens zur Frage gehören
    # und welche zum Kontext. Nur Kontext-Tokens dürfen später Antwort sein.
    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
    )

    offset_mapping = inputs.pop("offset_mapping")[0]
    sequence_ids = inputs.sequence_ids(0)

    with torch.no_grad():
        outputs = qa_model(**inputs)

    # Das QA-Modell bewertet jedes Token zweimal:
    # einmal als möglichen Anfang und einmal als mögliches Ende der Antwort.
    start_logits = outputs.start_logits[0].clone()
    end_logits   = outputs.end_logits[0].clone()

    # Frage, Sondertokens und Padding dürfen nicht Teil der Antwort sein.
    # Wir setzen sie auf -inf, damit ihre Wahrscheinlichkeit nach softmax 0 wird.
    for index, sequence_id in enumerate(sequence_ids):
        if sequence_id != 1:
            start_logits[index] = -torch.inf
            end_logits[index]   = -torch.inf

    start_probs = torch.softmax(start_logits, dim=0)
    end_probs   = torch.softmax(end_logits, dim=0)

    # Jede mögliche Antwortspanne besteht aus Start- und Endtoken.
    # Der Score ist hier das Produkt beider Wahrscheinlichkeiten,
    # analog zur früheren QuestionAnsweringPipeline.
    span_scores = torch.outer(start_probs, end_probs)
    num_tokens  = span_scores.shape[0]

    # Ungültige Spannen ausschließen:
    # - Ende darf nicht vor Anfang liegen.
    # - Antworten sollen nicht beliebig lang werden.
    starts_before_ends = torch.triu(torch.ones((num_tokens, num_tokens), dtype=torch.bool))
    short_enough = torch.tril(
        torch.ones((num_tokens, num_tokens), dtype=torch.bool),
        diagonal=max_answer_length - 1,
    )

    valid_spans = starts_before_ends & short_enough
    span_scores = span_scores.masked_fill(~valid_spans, 0)
    best_span   = torch.argmax(span_scores).item()

    best_start_token, best_end_token = divmod(best_span, num_tokens)

    # offset_mapping übersetzt Token-Positionen zurück auf Zeichenpositionen
    # im ursprünglichen Kontext. Dadurch bekommen wir wieder dieselbe Art
    # Ausgabe wie die alte Pipeline: answer, score, start, end.
    start_char, _ = offset_mapping[best_start_token].tolist()
    _, end_char = offset_mapping[best_end_token].tolist()

    return {
        "score": span_scores[best_start_token, best_end_token].item(),
        "start": start_char,
        "end": end_char,
        "answer": context[start_char:end_char],
    }

result = question_answerer(question="What is a good example of a question answering dataset?", context=context)
print(f"Answer: '{result['answer']}', score: {round(result['score'], 4)}, start: {result['start']}, end: {result['end']}")

Answer: 'SQuAD dataset', score: 0.5152, start: 147, end: 160


# Ein bisschen mehr low level - haben zwei Sequenzen dieselbe Bedeutung?

In [20]:
model_name = 'bert-base-cased-finetuned-mrpc'

## Tokenizer

In [21]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [22]:
tokens = tokenizer("Short")
tokens

{'input_ids': [101, 6373, 102], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1]}

In [23]:
# whole words can become a single token, CLS and SEP tokens are added
# CLS means classification, SEP means separation
for token in tokens.input_ids:
    print(f"{token} = {tokenizer.decode([token])}")

101 = [CLS]
6373 = Short
102 = [SEP]


In [24]:
# a sequence gets separated by [SEP] token
tokens = tokenizer("first", "second")
for token in tokens.input_ids:
    print(f"{token} = {tokenizer.decode([token])}")

101 = [CLS]
1148 = first
102 = [SEP]
1248 = second
102 = [SEP]


## Model

https://huggingface.co/bert-base-cased-finetuned-mrpc?library=true

### Trained on

Microsoft Research Paraphrase Corpus (MRPC) is a corpus consists of 5,801 sentence pairs collected from newswire articles. Each pair is labelled if it is a paraphrase or not by human annotators.

https://paperswithcode.com/dataset/mrpc



In [25]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [26]:
paraphrase = tokenizer(
    "The company HuggingFace is based in New York City",
    "HuggingFace's headquarters are situated in Manhattan",
    return_tensors="pt")
paraphrase_classification_logits = model(**paraphrase).logits
paraphrase_results = torch.softmax(paraphrase_classification_logits, dim=1).detach().numpy()[0]
print(f"Probability of being a paraphrase: {paraphrase_results[1]*100:.1f}%")


Probability of being a paraphrase: 90.5%


In [27]:
paraphrase = tokenizer(
    "The company HuggingFace is based in New York City",
    "Apples are especially bad for your health",
    return_tensors="pt")
paraphrase_classification_logits = model(**paraphrase).logits
paraphrase_results = torch.softmax(paraphrase_classification_logits, dim=1).detach().numpy()[0]
print(f"Probability of being a paraphrase: {paraphrase_results[1]*100:.1f}%")


Probability of being a paraphrase: 6.0%


# Deutsche Modelle

Deutsche einfache Transformer-Modelle sind leider etwas rar

Modelle
* Base: https://huggingface.co/bert-base-german-cased
* Squad: https://huggingface.co/deutsche-telekom/bert-multi-english-german-squad2

Data Sets
* https://tblock.github.io/10kGNAD/


In [28]:
from transformers import AutoModelForMaskedLM

# model_name = "distilbert-base-cased"
# model_name = "bert-base-german-cased"
# model_name = "bert-base-german-dbmdz-cased"
model_name = "bert-base-multilingual-cased"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
# sequence = f"Distilled models are smaller than the models they mimic. Using them instead of the large versions would help {tokenizer.mask_token} our carbon footprint."
# sequence = f"Deutschland ist ein tolles Land. Als Bürger hast du das Recht, {tokenizer.mask_token} zu tun was nicht gegen das Gesetz ist."
sequence = f"Deutschland ist ein tolles Land. Als Bürger hast du das Recht, {tokenizer.mask_token} zu tun."
sequence

'Deutschland ist ein tolles Land. Als Bürger hast du das Recht, [MASK] zu tun.'

In [30]:
tokens = tokenizer.encode(sequence)
for token in tokens:
    print(f"{token} = {tokenizer.decode([token])}")

101 = [CLS]
13011 = Deutschland
10298 = ist
10290 = ein
81754 = toll
10171 = ##es
12001 = Land
119 = .
11966 = Als
47473 = Bürger
10393 = has
10123 = ##t
10168 = du
10242 = das
31041 = Recht
117 = ,
103 = [MASK]
10304 = zu
53100 = tun
119 = .
102 = [SEP]


In [31]:
input = tokenizer.encode(sequence, return_tensors="pt")
mask_token_index = torch.where(input == tokenizer.mask_token_id)[1][0]
mask_token_index


tensor(16)

In [32]:
with torch.no_grad():
    token_logits = model(input).logits
token = token_logits[0, 1].argmax().item()
print(f"{token} = {tokenizer.decode([token])}")


13011 = Deutschland


In [33]:
mask_token_logits = token_logits[0, mask_token_index, :]
top_5 = torch.topk(mask_token_logits, 5)
probas = torch.softmax(top_5.values, dim=0).detach().numpy()
indices = top_5.indices.detach().numpy()
probas, indices


(array([0.8575853 , 0.04155795, 0.0363284 , 0.03400301, 0.03052528],
       dtype=float32),
 array([13011, 38451, 54760, 27879, 23942]))

In [34]:
for token, proba in zip(indices, probas):
    print(f"{token} = {tokenizer.decode([token])}, proba {proba*100:.1f}%")

13011 = Deutschland, proba 85.8%
38451 = nichts, proba 4.2%
54760 = Freiheit, proba 3.6%
27879 = Deutsch, proba 3.4%
23942 = etwas, proba 3.1%


In [35]:
for token in indices:
    print(sequence.replace(tokenizer.mask_token, tokenizer.decode([token])))

Deutschland ist ein tolles Land. Als Bürger hast du das Recht, Deutschland zu tun.
Deutschland ist ein tolles Land. Als Bürger hast du das Recht, nichts zu tun.
Deutschland ist ein tolles Land. Als Bürger hast du das Recht, Freiheit zu tun.
Deutschland ist ein tolles Land. Als Bürger hast du das Recht, Deutsch zu tun.
Deutschland ist ein tolles Land. Als Bürger hast du das Recht, etwas zu tun.
